# Task 3: Financial Data Analysis

This notebook contains the implementation of Task 3, which focuses on financial data retrieval and analysis. We'll use the `yfinance` library to fetch financial data for stocks mentioned in Reuters headlines.

## Overview
1. Financial Metrics Retrieval
2. Historical Data Analysis
3. Return Calculations
4. Peer Comparison

In [1]:
# Import required libraries
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. Financial Analyzer Class

The `FinancialAnalyzer` class is the core component that handles all financial data operations. It includes methods for:
- Fetching financial metrics
- Retrieving historical data
- Calculating returns
- Comparing peer companies

In [2]:
class FinancialAnalyzer:
    def __init__(self):
        """Initialize the financial analyzer"""
        self.cache = {}  # Cache for storing financial data

    def get_financial_metrics(self, ticker):
        """Get comprehensive financial metrics for a stock"""
        try:
            stock = yf.Ticker(ticker)
            info = stock.info
            
            metrics = {
                'Basic Info': {
                    'Company Name': info.get('longName', 'N/A'),
                    'Sector': info.get('sector', 'N/A'),
                    'Industry': info.get('industry', 'N/A'),
                },
                'Market Data': {
                    'Market Cap': info.get('marketCap', 'N/A'),
                    'Current Price': info.get('currentPrice', 'N/A'),
                    'Volume': info.get('volume', 'N/A'),
                    '52 Week High': info.get('fiftyTwoWeekHigh', 'N/A'),
                    '52 Week Low': info.get('fiftyTwoWeekLow', 'N/A'),
                },
                'Valuation Metrics': {
                    'P/E Ratio': info.get('trailingPE', 'N/A'),
                    'Forward P/E': info.get('forwardPE', 'N/A'),
                    'PEG Ratio': info.get('pegRatio', 'N/A'),
                    'Price to Book': info.get('priceToBook', 'N/A'),
                    'Enterprise Value': info.get('enterpriseValue', 'N/A'),
                },
                'Financial Metrics': {
                    'Revenue': info.get('totalRevenue', 'N/A'),
                    'Profit Margin': info.get('profitMargins', 'N/A'),
                    'Operating Margin': info.get('operatingMargins', 'N/A'),
                    'Return on Equity': info.get('returnOnEquity', 'N/A'),
                    'Quarterly Growth': info.get('revenueQuarterlyGrowth', 'N/A'),
                },
                'Trading Info': {
                    'Beta': info.get('beta', 'N/A'),
                    'Short Ratio': info.get('shortRatio', 'N/A'),
                    'Dividend Yield': info.get('dividendYield', 'N/A'),
                    'Average Volume': info.get('averageVolume', 'N/A'),
                }
            }
            
            self._format_metrics(metrics)
            return metrics, None
            
        except Exception as e:
            return None, f"Error fetching metrics for {ticker}: {str(e)}"

    def _format_metrics(self, metrics):
        """Format metrics for better readability"""
        if isinstance(metrics['Market Data']['Market Cap'], (int, float)):
            metrics['Market Data']['Market Cap'] = f"${metrics['Market Data']['Market Cap']/1e9:.2f}B"
        
        for category in metrics.values():
            for key, value in category.items():
                if isinstance(value, float) and 'Ratio' not in key and 'Price' not in key:
                    if abs(value) < 1:
                        category[key] = f"{value*100:.2f}%"

    def get_historical_data(self, ticker, period='1y', interval='1d'):
        """Get historical price data with error handling and caching"""
        cache_key = f"{ticker}_{period}_{interval}"
        
        if cache_key in self.cache:
            return self.cache[cache_key], None

        try:
            stock = yf.Ticker(ticker)
            hist_data = stock.history(period=period, interval=interval)
            
            if hist_data.empty:
                return None, f"No historical data available for {ticker}"
            
            hist_data['Daily_Return'] = hist_data['Close'].pct_change()
            hist_data['Volatility'] = hist_data['Daily_Return'].rolling(window=20).std()
            hist_data['MA20'] = hist_data['Close'].rolling(window=20).mean()
            hist_data['MA50'] = hist_data['Close'].rolling(window=50).mean()
            
            self.cache[cache_key] = hist_data
            return hist_data, None
            
        except Exception as e:
            return None, f"Error fetching historical data for {ticker}: {str(e)}"

    def calculate_returns(self, prices):
        """Calculate comprehensive return metrics"""
        if len(prices) < 2:
            return None
        
        daily_returns = prices['Close'].pct_change()
        
        return {
            'Return Metrics': {
                'Daily Average Return': f"{daily_returns.mean()*100:.2f}%",
                'Daily Std Dev': f"{daily_returns.std()*100:.2f}%",
                'Annualized Return': f"{daily_returns.mean() * 252 * 100:.2f}%",
                'Annualized Volatility': f"{daily_returns.std() * np.sqrt(252) * 100:.2f}%",
                'Total Return': f"{((prices['Close'].iloc[-1] / prices['Close'].iloc[0]) - 1) * 100:.2f}%",
                'Sharpe Ratio': f"{(daily_returns.mean() / daily_returns.std()) * np.sqrt(252):.2f}",
                'Max Drawdown': f"{self._calculate_max_drawdown(prices['Close'])*100:.2f}%"
            }
        }

    def _calculate_max_drawdown(self, prices):
        """Calculate maximum drawdown from peak"""
        peak = prices.expanding(min_periods=1).max()
        drawdown = (prices - peak) / peak
        return drawdown.min()

    def get_peer_comparison(self, ticker, peers=None):
        """Compare financial metrics with peer companies"""
        try:
            if peers is None:
                stock = yf.Ticker(ticker)
                peers = stock.info.get('recommendationKey', [])[:5]
            
            comparison_data = []
            for company in [ticker] + peers:
                metrics, error = self.get_financial_metrics(company)
                if metrics and not error:
                    comparison_data.append({
                        'Ticker': company,
                        'Market Cap': metrics['Market Data']['Market Cap'],
                        'P/E Ratio': metrics['Valuation Metrics']['P/E Ratio'],
                        'Profit Margin': metrics['Financial Metrics']['Profit Margin'],
                        'Beta': metrics['Trading Info']['Beta']
                    })
            
            return pd.DataFrame(comparison_data), None
            
        except Exception as e:
            return None, f"Error in peer comparison: {str(e)}"

## 2. Example Usage

Let's demonstrate how to use the FinancialAnalyzer class with some example stocks.

In [3]:
# Initialize the analyzer
analyzer = FinancialAnalyzer()

# Example stock ticker
ticker = 'AAPL'

# Get financial metrics
metrics, error = analyzer.get_financial_metrics(ticker)
if metrics:
    print("\nFinancial Metrics for", ticker)
    for category, data in metrics.items():
        print(f"\n{category}:")
        for key, value in data.items():
            print(f"{key}: {value}")


Financial Metrics for AAPL

Basic Info:
Company Name: Apple Inc.
Sector: Technology
Industry: Consumer Electronics

Market Data:
Market Cap: $3324.13B
Current Price: 221.05
Volume: 18144804
52 Week High: 260.1
52 Week Low: 164.08

Valuation Metrics:
P/E Ratio: 36.416805
Forward P/E: 26.7117
PEG Ratio: N/A
Price to Book: 58.68065
Enterprise Value: 3530225090560

Financial Metrics:
Revenue: 391034994688
Profit Margin: 23.97%
Operating Margin: 31.17%
Return on Equity: 1.5741299
Quarterly Growth: N/A

Trading Info:
Beta: 1.24
Short Ratio: 3.37
Dividend Yield: 0.43%
Average Volume: 46061296


In [4]:
# Get historical data and plot
hist_data, error = analyzer.get_historical_data(ticker, period='1y')
if hist_data is not None:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        vertical_spacing=0.03,
                        row_heights=[0.7, 0.3])
    
    fig.add_trace(go.Scatter(x=hist_data.index, y=hist_data['Close'],
                            name='Close Price'),
                row=1, col=1)
    
    fig.add_trace(go.Bar(x=hist_data.index, y=hist_data['Volume'],
                        name='Volume'),
                row=2, col=1)
    
    fig.update_layout(height=600, title_text=f"{ticker} Price and Volume")
    fig.show()

In [5]:
# Calculate returns
returns = analyzer.calculate_returns(hist_data)
if returns:
    print("\nReturn Metrics:")
    for category, metrics in returns.items():
        print(f"\n{category}:")
        for key, value in metrics.items():
            print(f"{key}: {value}")


Return Metrics:

Return Metrics:
Daily Average Return: 0.06%
Daily Std Dev: 1.46%
Annualized Return: 16.37%
Annualized Volatility: 23.13%
Total Return: 14.57%
Sharpe Ratio: 0.71
Max Drawdown: -15.35%


In [6]:
# Peer comparison
peers = ['MSFT', 'GOOGL', 'META', 'AMZN']
comparison_df, error = analyzer.get_peer_comparison(ticker, peers)
if comparison_df is not None:
    print("\nPeer Comparison:")
    display(comparison_df)


Peer Comparison:


,Ticker,Market Cap,P/E Ratio,Profit Margin,Beta
0,AAPL,$3324.13B,36.416805,23.97%,1.24
1,MSFT,$3168.15B,35.216526,35.61%,90.40%
2,GOOGL,$2443.94B,26.349669,27.74%,1.034
3,META,$1554.05B,29.037264,35.55%,1.215
4,AMZN,$2397.42B,48.717950,8.04%,1.146


## 3. Additional Notes

1. The `FinancialAnalyzer` class uses caching to improve performance when fetching historical data multiple times.
2. All methods include error handling to gracefully handle API failures or invalid data.
3. The financial metrics are automatically formatted for better readability (e.g., converting to percentages, billions for market cap).
4. The historical data analysis includes technical indicators like moving averages and volatility.
5. The return calculations include important metrics like Sharpe Ratio and Maximum Drawdown.